In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Cipta plugin transpiler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
```
</details>
Mencipta [plugin transpiler](transpiler-plugins) adalah cara yang bagus untuk berkongsi kod transpilasi anda dengan komuniti Qiskit yang lebih luas, membolehkan pengguna lain memanfaatkan fungsi yang telah anda bangunkan. Terima kasih atas minat anda untuk menyumbang kepada komuniti Qiskit!

Sebelum mencipta plugin transpiler, anda perlu menentukan jenis plugin yang sesuai untuk situasi anda. Terdapat tiga jenis plugin transpiler:
- [**Plugin peringkat Transpiler**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Pilih ini jika anda mentakrifkan pengurus laluan yang boleh menggantikan salah satu daripada [6 peringkat](transpiler-stages) pengurus laluan berperingkat praset.
- [**Plugin sintesis unitar**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Pilih ini jika kod transpilasi anda mengambil matriks unitar (diwakili sebagai tatasusunan Numpy) sebagai input dan menghasilkan penerangan Circuit kuantum yang melaksanakan unitar tersebut.
- [**Plugin sintesis peringkat tinggi**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Pilih ini jika kod transpilasi anda mengambil "objek peringkat tinggi" seperti operator Clifford atau fungsi linear sebagai input dan menghasilkan penerangan Circuit kuantum yang melaksanakan objek peringkat tinggi tersebut. Objek peringkat tinggi diwakili oleh subkelas kelas [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Setelah menentukan jenis plugin yang hendak dicipta, ikut langkah-langkah berikut untuk mencipta plugin:

1. Cipta subkelas kelas plugin abstrak yang sesuai:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) untuk plugin peringkat transpiler,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) untuk plugin sintesis unitar, dan
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) untuk plugin sintesis peringkat tinggi.
2. Dedahkan kelas sebagai [titik masuk setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) dalam metadata pakej, biasanya dengan mengedit fail `pyproject.toml`, `setup.cfg`, atau `setup.py` untuk pakej Python anda.

Tiada had bilangan plugin yang boleh ditakrifkan oleh satu pakej, tetapi setiap plugin mesti mempunyai nama yang unik. SDK Qiskit sendiri menyertakan beberapa plugin, yang namanya turut dikhaskan. Nama yang dikhaskan ialah:

- Plugin peringkat transpiler: Lihat [jadual ini](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Plugin sintesis unitar: `default`, `aqc`, `sk`
- Plugin sintesis peringkat tinggi:

| Kelas operasi | Nama operasi | Nama dikhaskan |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

Dalam bahagian seterusnya, kami menunjukkan contoh langkah-langkah ini untuk pelbagai jenis plugin. Dalam contoh-contoh ini, kami andaikan bahawa kami sedang mencipta pakej Python bernama `my_qiskit_plugin`. Untuk maklumat tentang mencipta pakej Python, anda boleh semak [tutorial ini](https://packaging.python.org/en/latest/tutorials/packaging-projects/) dari laman web Python.
## Contoh: Cipta plugin peringkat transpiler
Dalam contoh ini, kami mencipta plugin peringkat transpiler untuk peringkat `layout` (lihat [Peringkat Transpiler](transpiler-stages) untuk penerangan 6 peringkat saluran paip transpilasi terbina dalam Qiskit).
Plugin kami hanya menjalankan [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) untuk beberapa percubaan yang bergantung pada tahap pengoptimuman yang diminta.

Pertama, kami mencipta subkelas [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). Terdapat satu kaedah yang perlu dilaksanakan, iaitu [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Kaedah ini mengambil [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) sebagai input dan mengembalikan pengurus laluan yang kami takrifkan. Objek PassManagerConfig menyimpan maklumat tentang Backend sasaran, seperti peta gandingannya dan Gate asas.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Sekarang, kami mendedahkan plugin dengan menambah titik masuk dalam metadata pakej Python kami.
Di sini, kami andaikan bahawa kelas yang kami takrifkan didedahkan dalam modul bernama `my_qiskit_plugin`, contohnya dengan diimport dalam fail `__init__.py` modul `my_qiskit_plugin`.
Kami mengedit fail `pyproject.toml`, `setup.cfg`, atau `setup.py` pakej kami (bergantung pada jenis fail yang anda pilih untuk menyimpan metadata projek Python anda):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Lihat [jadual peringkat plugin transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) untuk titik masuk dan jangkaan bagi setiap peringkat transpiler.

Untuk menyemak sama ada plugin anda berjaya dikesan oleh Qiskit, pasang pakej plugin anda dan ikut arahan di [Plugin transpiler](transpiler-plugins#list-available-transpiler-stage-plugins) untuk menyenaraikan plugin yang dipasang, dan pastikan plugin anda muncul dalam senarai:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Jika plugin contoh kami dipasang, nama `my_layout` akan muncul dalam senarai ini.

Jika anda ingin menggunakan peringkat transpiler terbina dalam sebagai titik permulaan untuk plugin peringkat transpiler anda, anda boleh mendapatkan pengurus laluan untuk peringkat transpiler terbina dalam menggunakan [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). Sel kod berikut menunjukkan cara untuk mendapatkan peringkat pengoptimuman terbina dalam untuk tahap pengoptimuman 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Contoh: Cipta plugin sintesis unitar
Dalam contoh ini, kami akan mencipta plugin sintesis unitar yang hanya menggunakan laluan transpilasi terbina dalam [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) untuk mensintesis Gate. Sudah tentu, plugin anda sendiri akan melakukan sesuatu yang lebih menarik daripada itu.

Kelas [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) mentakrifkan antara muka dan kontrak untuk plugin sintesis unitar. Kaedah utamanya ialah
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
yang mengambil tatasusunan Numpy yang menyimpan matriks unitar sebagai input
dan mengembalikan [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) yang mewakili Circuit yang disintesis daripada matriks unitar tersebut.
Selain kaedah `run`, terdapat beberapa kaedah sifat yang perlu ditakrifkan.
Lihat [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) untuk dokumentasi semua sifat yang diperlukan.

Mari cipta subkelas UnitarySynthesisPlugin kami:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Jika anda mendapati input yang tersedia untuk kaedah [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
tidak mencukupi untuk tujuan anda, sila [buka isu](https://github.com/Qiskit/qiskit/issues/new/choose) menerangkan keperluan anda. Perubahan pada antara muka plugin, seperti menambah input pilihan tambahan, akan dilakukan dengan cara yang serasi ke belakang supaya tidak memerlukan perubahan daripada plugin sedia ada.

> **Note:** Semua kaedah yang diawali dengan `supports_` dikhaskan pada kelas terbitan `UnitarySynthesisPlugin` sebagai sebahagian daripada antara muka. Anda tidak seharusnya mentakrifkan sebarang kaedah `supports_*` tersuai pada subkelas yang tidak ditakrifkan dalam kelas abstrak.

Sekarang, kami mendedahkan plugin dengan menambah titik masuk dalam metadata pakej Python kami.
Di sini, kami andaikan bahawa kelas yang kami takrifkan didedahkan dalam modul bernama `my_qiskit_plugin`, contohnya dengan diimport dalam fail `__init__.py` modul `my_qiskit_plugin`.
Kami mengedit fail `pyproject.toml`, `setup.cfg`, atau `setup.py` pakej kami:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Seperti sebelumnya, jika projek anda menggunakan `setup.cfg` atau `setup.py` dan bukannya `pyproject.toml`, lihat [dokumentasi setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) untuk cara menyesuaikan baris ini dengan situasi anda.

Untuk menyemak sama ada plugin anda berjaya dikesan oleh Qiskit, pasang pakej plugin anda dan ikut arahan di [Plugin transpiler](transpiler-plugins#list-available-unitary-synthesis-plugins) untuk menyenaraikan plugin yang dipasang, dan pastikan plugin anda muncul dalam senarai:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Jika plugin contoh kami dipasang, nama `my_unitary_synthesis` akan muncul dalam senarai ini.

Untuk menampung plugin sintesis unitar yang mendedahkan pelbagai pilihan,
antara muka plugin mempunyai pilihan untuk pengguna menyediakan
kamus konfigurasi bebas bentuk. Ini akan dihantar ke kaedah `run`
melalui argumen kata kunci `options`. Jika plugin anda mempunyai pilihan konfigurasi ini, anda seharusnya mendokumentasikannya dengan jelas.

## Contoh: Cipta plugin sintesis peringkat tinggi

Dalam contoh ini, kami akan mencipta plugin sintesis peringkat tinggi yang hanya menggunakan fungsi terbina dalam [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) untuk mensintesis operator Clifford.

Kelas [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) mentakrifkan antara muka dan kontrak untuk plugin sintesis peringkat tinggi. Kaedah utamanya ialah [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
Argumen kedudukan `high_level_object` ialah [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) yang mewakili objek "peringkat tinggi" yang hendak disintesis. Contohnya, ia boleh menjadi
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) atau
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Argumen kata kunci berikut hadir:
- `target` menentukan Backend sasaran, membolehkan plugin
mengakses semua maklumat khusus sasaran,
seperti peta gandingan, set Gate yang disokong, dan sebagainya
- `coupling_map` hanya menentukan peta gandingan, dan hanya digunakan apabila `target` tidak dinyatakan.
- `qubits` menentukan senarai Qubit yang menjadi asas
objek peringkat tinggi ditakrifkan, sekiranya sintesis dilakukan pada Circuit fizikal.
Nilai ``None`` menunjukkan bahawa susun atur belum dipilih dan Qubit fizikal dalam sasaran atau peta gandingan yang digunakan oleh operasi ini belum ditentukan.
- `options`, kamus konfigurasi bebas bentuk untuk pilihan khusus plugin. Jika plugin anda mempunyai pilihan konfigurasi ini anda
seharusnya mendokumentasikannya dengan jelas.

Kaedah `run` mengembalikan [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
yang mewakili Circuit yang disintesis daripada objek peringkat tinggi tersebut.
Ia juga dibenarkan untuk mengembalikan `None`, menunjukkan bahawa plugin tidak dapat mensintesis objek peringkat tinggi yang diberikan.
Sintesis sebenar objek peringkat tinggi dilakukan oleh laluan transpiler
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

Selain kaedah `run`, terdapat beberapa kaedah sifat yang perlu ditakrifkan.
Lihat [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) untuk dokumentasi semua sifat yang diperlukan.

Mari takrifkan subkelas HighLevelSynthesisPlugin kami: